## Data analysis & visualization: LAB 4

**Part 1**

Worth noting (to the teacher(s)): I already coded along during todays lesson and syncronized that notebook to github. I'm still chosing part 1 as I feel the need to solidify some concepts.

Also a small joke: Does it have a positronic brain?

Ref: Star Trek - The Next Generation

Explaining this dry joke: The question is asked about the data.

In the referenced series, Data, is an android (like a "passable" humanoid robot), one of the main characters.

Task 1:8

In [ ]:
import pandas as pd

loans_raw = pd.read_csv('data/library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])

members_raw = pd.read_csv('data/library_members.csv')

In [ ]:
loans = loans_raw.copy()

print(loans.shape)

loans.dtypes

In [ ]:
loans.isna().sum()

In [ ]:
(loans.isna().mean() * 100).round(1)

**Answer for task 1 (above (^) this MD): Two columns have gaps, they are loan_date and return_date. Missing vales are 3 (0.5%) and 36 (6%) for both columns (in the order provided).**
___

In [ ]:
loans[loans['loan_date'].isna()]

In [ ]:
not_returned = loans[loans['return_date'].isna() & (loans['returned'] == False)] # Here we are querying for reasonable data, where the books has not been returned, looks valid.

not_returned.shape
# not_back

In [ ]:
returned_but_date_missing = loans[
    loans['return_date'].isna()
    & (loans['returned'] == True)
]
# Now we are querying for data where returned is true but the return date is missing. Here we are observing a contradiction in the data.

returned_but_date_missing.shape
# returned_but_date_missing

In [ ]:
loans['has_return_date'] = loans['return_date'].notna() # Creating a column for items (books) that has a return date, or more specifically values that are not missing from the return_date column. notcha() makes has_return_date a bool.

loans['has_return_date'].dtype

In [ ]:
loans['has_return_date'].value_counts() # 600 in total checks out with the shape of the DF.

**Answer for task 2 (above (^) this MD): 30 missing return dates are perfectly reasonable, while 6 books are registered as returned with a missing return date. The latter (6) suggests these are missing by accident**
___

In [ ]:
members = members_raw.copy()

members.shape

In [ ]:
members.info()

In [ ]:
members['birth_year'].describe()

In [ ]:
members['birth_year'].value_counts().loc[[1900, 2030]]

In [ ]:
members[members['birth_year'].isin([1900, 2030])]

In [ ]:
print(members.duplicated().sum()) # Checking for duplicate whole rows.

In [ ]:
members[members.duplicated(keep=False)].sort_values('member_id') # Here the output gives a clearer view of rows that are identical over all (3) of the columns.

In [ ]:
print(members['member_id'].duplicated().sum())

**Answer for task 3 (above (^) this MD):**
- **branch is missing in 12 rows.**
- **birth_year is missing in 18 rows.**
- **8 members have the invalid placeholder year 1900.**
- **3 members have the impossible future year 2030.**
- **4 rows are duplicates (4 pairs).**
- **4 member_id values are duplicated because of those repeated rows.**

In [ ]:
members['birth_year'] = members['birth_year'].where(members['birth_year'].between(1920, 2015))

In [ ]:
print(members['birth_year'].isna().sum())

In [ ]:
members['birth_year'].describe().round(2)

**Answer for task 4 (above (^) this MD): After treating 1900 and 2030 as invalid, birth_year has 29 missing values. The observed range is now 1957–2010, and the mean increases because the eight invalid 1900 values (vs 3 2030) had pulled the average (mean) downward.**
___

**5a - Remove**

In [ ]:
members.dropna().shape # Removes entire rows if any column value is missing.

In [ ]:
members.dropna(subset=['branch']).shape # Remove rows when missing in the branch column and ignores missing values in other columns. (148 - 12 = 136)

In [ ]:
members.dropna(subset=['birth_year']).shape

**5b - Fill missing branches**

In [ ]:
members['branch'] = members['branch'].fillna('Unknown')

In [ ]:
members['branch'].value_counts()

**5c - Test median filling separately**

In [ ]:
birth_year_median = members['birth_year'].median()

print(birth_year_median)

In [ ]:
birth_year_filled = members['birth_year'].fillna(birth_year_median)

print(members['birth_year'].isna().sum())

In [ ]:
print(birth_year_filled.isna().sum())

In [ ]:
print(members['birth_year'].mean().round(2))

In [ ]:
print(birth_year_filled.mean().round(2))

In [ ]:
print(members['birth_year'].std().round(2))


In [ ]:
print(birth_year_filled.std().round(2))

**Answer for task 5 (above (^) this MD): Filling birth_year with the median changes the mean very little, but reduces the standard deviation because all missing values are replaced with the same central value. I will therefore leave the unknown birth years missing rather than inventing years.**

**Filling missing values can preserve rows, but it can also alter the distribution of the data.**
___

In [ ]:
print("Loans:", loans.shape)
print("Members:", members.shape)

In [ ]:
late_fee_before = loans['late_fee'].sum()
print("Late fee total before merge:", late_fee_before)

In [ ]:
merged_bad = loans.merge(members, on='member_id', how='left') # Using the merge method on the loans DataFrame (left), passing members DF, using member_id as the matching key (at the index level).

In [ ]:
merged_bad.shape

In [ ]:
late_fee_after = merged_bad['late_fee'].sum()

print('Before:', late_fee_before)
print('After:', late_fee_after)
print('Difference:', late_fee_after - late_fee_before)

In [ ]:
merged_checked = loans.merge(
    members,
    on='member_id',
    how='left',
    validate='many_to_one'
)

**Answer for task 6 (above (^) this MD): The unvalidated merge creates 616 rows instead of 600 because duplicate member records cause some loans to match more than once. This also overcounts late fees by 237.5 kr. The many-to-one validation correctly raises an error because member_id is not unique in the members table.**

In [ ]:
members = (
    members
    .drop_duplicates()
    .reset_index(drop=True)
)

members.shape

In [ ]:
print(members['member_id'].duplicated().sum())

# members['member_id'].is_unique # Also works for this step. True expected.

In [ ]:
merged = loans.merge(
    members,
    on='member_id',
    how='left',
    validate='many_to_one',
    indicator=True
)

In [ ]:
merged.shape

In [ ]:
print(merged['late_fee'].sum()) # Now matches the total before the merge. 7487.5 expected.

In [ ]:
merged['_merge'].value_counts()

In [ ]:
missing_members = merged[
    merged['_merge'] == 'left_only'
]

missing_members

In [ ]:
missing_members['member_id'].nunique()

In [ ]:
missing_members['member_id'].value_counts()

In [ ]:
print(missing_members['late_fee'].sum())

In [ ]:
merged_inner = loans.merge(
    members,
    on='member_id',
    how='inner',
    validate='many_to_one'
)

print(merged_inner.shape)
print(merged_inner['late_fee'].sum())

**Answer for task 7 (above (^) this MD): After removing four duplicate member rows, member_id is unique and the validated left merge keeps all 600 loans without changing the late-fee total. Thirty-one loans from six distinct member IDs have no record in the member DataFrame. An inner merge would discard those loans and their 347.5 kr in late fees.**